# 02 · Seq2seq + attention from scratch

A tiny encoder-decoder with multi-head cross-attention, trained to turn linearised specs into descriptions — the architecture family behind the production T5 (`app/generator.py`), with hand-written beam search.

In [ ]:
# --- bootstrap -------------------------------------------------------------
# Working-directory independent, so this notebook runs identically in Jupyter,
# in VS Code and in Google Colab (kernel CWD is /content, and the checkout may
# sit anywhere below it).  Finds the repository, puts services/model-server on
# sys.path, and generates the git-ignored synthetic corpus on first use.
#
# Needs the *repository*, not just this .ipynb: it imports `app` and reads
# `ml/data/generated`.  In a fresh Colab runtime run this once, above or below
# this cell, then re-run:
#
#     !git clone <your-fork-or-url> /content/catalogue-ai
#
# Already cloned elsewhere? Either set CATALOGUE_AI_ROOT=/path/to/catalogue-ai,
# or rely on the scan below.
import os, sys, pathlib

_HINT = pathlib.Path("ml/notebooks/nbsetup.py")


def _is_repo(path):
    return (path / _HINT).is_file()


def _candidates():
    """Ordered places a checkout might be, nearest first."""
    out = []
    env = os.environ.get("CATALOGUE_AI_ROOT")
    if env:
        out.append(pathlib.Path(env).expanduser())
    cwd = pathlib.Path.cwd().resolve()
    out.extend([cwd, *cwd.parents])
    out.extend([pathlib.Path("/content"), pathlib.Path.home()])
    seen, ordered = set(), []
    for p in out:
        try:
            r = p.resolve()
        except OSError:
            continue
        if r not in seen:
            seen.add(r)
            ordered.append(r)
    return ordered


def _find_repo():
    for c in _candidates():
        if _is_repo(c):
            return c
    # shallow scan: a Colab clone can land in any folder below /content,
    # including below a Drive mount
    frontier = [pathlib.Path("/content"), pathlib.Path.home(),
                pathlib.Path.cwd().resolve()]
    for _ in range(4):
        nxt = []
        for d in frontier:
            if _is_repo(d):
                return d
            try:
                kids = [p for p in sorted(d.iterdir())
                        if p.is_dir() and not p.name.startswith(".")
                        and p.name != "__pycache__"]
            except (OSError, PermissionError):
                kids = []
            nxt.extend(kids)
        frontier = nxt
        if not frontier:
            break
    return None


REPO_ROOT = _find_repo()
if REPO_ROOT is None:
    raise SystemExit(
        "catalogue-ai repository not found.\n\n"
        "This notebook needs the repository (it imports `app` and reads\n"
        "ml/data/generated), not just this .ipynb file. In a fresh Colab\n"
        "runtime run this once, then re-run this cell:\n\n"
        "    !git clone <your-fork-or-url> /content/catalogue-ai\n\n"
        "Already cloned elsewhere? Point the notebook at it:\n\n"
        "    import os; os.environ['CATALOGUE_AI_ROOT'] = '/path/to/catalogue-ai'\n\n"
        "searched:\n  " + "\n  ".join(str(c) for c in _candidates()))

sys.path.insert(0, str(REPO_ROOT / "ml" / "notebooks"))
import nbsetup

_summary = nbsetup.bootstrap()
REPO_ROOT = pathlib.Path(_summary["repo_root"])
DATA_DIR = pathlib.Path(_summary["data_dir"])

In [ ]:
import json
import numpy as np
import torch
import matplotlib.pyplot as plt
from app.linearise import linearise_spec
from app.from_scratch.attention import EncoderBlock, DecoderBlock
from app.from_scratch.padding import pad_batch, causal_key_mask, encoder_mask
from app.from_scratch.beam_search import beam_search

CONFIG = dict(d_model=64, n_heads=4, d_ff=128, epochs=8, batch=16,
              lr=3e-3, seed=42, max_src=48, max_tgt=40)
torch.manual_seed(CONFIG['seed'])
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## Corpus and a padding-aware batcher

`torch.tensor([[3, 7, 9], [4, 5]])` raises `ValueError: expected sequence of length 3 at dim 1`. Specs and descriptions tokenise to different lengths, so a batch has to be padded **and** masked: the encoder must not attend to padding, the decoder needs causal *and* padding masks, and the loss must ignore `<pad>` targets.

In [ ]:
PAD, BOS, EOS = 0, 1, 2

corpus = [json.loads(l) for l in (DATA_DIR / 'train.jsonl').open()][:800]
vocab = {'<pad>': PAD, '<bos>': BOS, '<eos>': EOS}
for r in corpus:
    for tok in (linearise_spec(r['spec']) + ' ' + r['description']).lower().split():
        vocab.setdefault(tok, len(vocab))
inv = {v: k for k, v in vocab.items()}

def encode(text, limit):
    return [vocab[t] for t in text.lower().split() if t in vocab][:limit]

def make_batch(rows):
    """Pad a ragged batch and return it with its padding masks."""
    src, src_mask = pad_batch(
        [encode(linearise_spec(r['spec']), CONFIG['max_src']) for r in rows], PAD)
    tgt, tgt_mask = pad_batch(
        [[BOS] + encode(r['description'], CONFIG['max_tgt']) + [EOS] for r in rows], PAD)
    return src, src_mask, tgt, tgt_mask

print('vocab:', len(vocab))

In [ ]:
class TinySeq2Seq(torch.nn.Module):
    def __init__(self, n_vocab):
        super().__init__()
        d, h, f = CONFIG['d_model'], CONFIG['n_heads'], CONFIG['d_ff']
        self.emb = torch.nn.Embedding(n_vocab, d, padding_idx=PAD)
        self.enc = EncoderBlock(d, h, f)
        self.dec = DecoderBlock(d, h, f)
        self.out = torch.nn.Linear(d, n_vocab)

    def forward(self, src, src_mask, tgt, tgt_mask, return_weights=False):
        mem = self.enc(self.emb(src), mask=encoder_mask(src_mask))
        x, attn = self.dec(self.emb(tgt), mem,
                           tgt_mask=causal_key_mask(tgt_mask),
                           memory_mask=encoder_mask(src_mask),
                           return_weights=True)
        logits = self.out(x)
        return (logits, attn) if return_weights else logits

model = TinySeq2Seq(len(vocab)).to(device)
print('parameters:', sum(p.numel() for p in model.parameters()))

In [ ]:
# teacher-forced training on (spec -> description)
opt = torch.optim.Adam(model.parameters(), lr=CONFIG['lr'])
losses = []
for epoch in range(CONFIG['epochs']):
    order = torch.randperm(len(corpus)).tolist()
    epoch_losses = []
    for start in range(0, len(order) - CONFIG['batch'] + 1, CONFIG['batch']):
        rows = [corpus[i] for i in order[start:start + CONFIG['batch']]]
        src, src_mask, tgt, tgt_mask = (t.to(device) for t in make_batch(rows))
        logits = model(src, src_mask, tgt[:, :-1], tgt_mask[:, :-1])
        loss = torch.nn.functional.cross_entropy(
            logits.reshape(-1, len(vocab)), tgt[:, 1:].reshape(-1),
            ignore_index=PAD)
        opt.zero_grad(); loss.backward(); opt.step()
        epoch_losses.append(float(loss))
    losses.append(float(np.mean(epoch_losses)))
    print(json.dumps({'epoch': epoch, 'loss': round(losses[-1], 4)}))

plt.figure(figsize=(5, 3.2))
plt.plot(losses, marker='o'); plt.xlabel('epoch'); plt.ylabel('loss')
plt.title('teacher-forced cross-entropy'); plt.tight_layout(); plt.show()

## Attention maps: which spec fields does the decoder look at?

In [ ]:
demo = corpus[0]
src, src_mask, tgt, tgt_mask = (t.to(device) for t in make_batch([demo]))
with torch.no_grad():
    _, attn = model(src, src_mask, tgt[:, :12], tgt_mask[:, :12],
                     return_weights=True)
print('spec   :', linearise_spec(demo['spec']))
print('target :', demo['description'])
plt.figure(figsize=(9, 4))
plt.imshow(attn[0, 0].cpu(), aspect='auto')
plt.xlabel('spec tokens'); plt.ylabel('output steps')
plt.title('cross-attention (head 0)'); plt.tight_layout(); plt.show()

## Hand-written beam search decoding

The step function masks padding exactly as the training forward pass does, otherwise the decoded prefix — which contains no `<pad>` — would build a causal mask of the wrong width. `<pad>` is also forbidden outright: a decoder must never *emit* padding, and left unconstrained it is the single most likely token, so the beams collapse to empty strings.

In [ ]:
@torch.no_grad()
def step_fn(prefix):
    tgt = torch.tensor([list(prefix)], device=device)
    mem = model.enc(model.emb(src), mask=encoder_mask(src_mask))
    # DecoderBlock returns a bare tensor unless return_weights=True
    x = model.dec(model.emb(tgt), mem,
                  tgt_mask=causal_key_mask(tgt != PAD),
                  memory_mask=encoder_mask(src_mask))
    logp = torch.log_softmax(model.out(x)[0, -1], dim=-1)
    logp[PAD] = float('-inf')  # never emit padding
    return {i: float(logp[i]) for i in range(len(vocab)) if i != PAD}

beams = beam_search(step_fn, bos=BOS, eos=EOS,
                    beam_width=4, max_len=24, num_return=3)
for b in beams:
    print(round(b.score, 3), '|', ' '.join(inv[t] for t in b.tokens[1:-1]))
print()
print('reference:', demo['description'])